In [2]:
import scanpy as sc
from run_dgcna_correlation import run_dgcna_analysis
from run_dgcna_reference import run_dgcna_reference
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


In [3]:

adata=sc.read_loom('Integrated_betas.loom') 
adata.obs_names=adata.obs.obs_names
adata.var_names=adata.var.gene_name


C:\Users\Leo\anaconda3\Lib\functools.py:934: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
C:\Users\Leo\anaconda3\Lib\functools.py:934: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


In [4]:
sel_genes=pd.read_excel("sel_genes.xlsx")
ndcg_genes=sel_genes.Gene_name.unique()
random_genes = np.random.choice(ndcg_genes, size=10, replace=False)

In [21]:
results = run_dgcna_analysis(
    adata, 
    genes=ndcg_genes,  # optional: subset to specific genes
    disease_col='Disease',
    t2d_value='T2D',
    ctrl_value='normal',
    n_bootstrap=1,
    threshold=0.975
)

Using 664 / 671 specified genes

=== Step 1: Computing LMM residuals (per condition) ===
Cells: 582, donors: 9, genes: 664
Cells: 456, donors: 8, genes: 664

=== Step 2: Building differential network ===
T2D cells: 582, Control cells: 456

=== Step 3: Bootstrap filtering ===


Random donor groups (with replacement): 100%|████████████████████████████████████████████| 1/1 [00:00<00:00,  5.52it/s]

Significant edges: 165212 / 220116 (75.1%)


In [37]:
# Generate Expression and Obs Dataframes to test with the R implementation

valid_genes=[i for i in ndcg_genes if i in adata.var_names]
pd.DataFrame(adata[:,valid_genes].X.toarray(), index=adata[:,valid_genes].obs_names, columns=adata[:,valid_genes].var_names).to_csv('Expr_beta.csv')
adata[:,valid_genes].obs[['Donor','Disease']].to_csv('Obs_beta.csv')

In [38]:
# Read in the residuals

R_residuals_normal=pd.read_csv('R_residuals_normal.csv', index_col=0)
R_residuals_T2D=pd.read_csv('R_residuals_T2D.csv', index_col=0)
R_residuals=pd.concat([R_residuals_normal, R_residuals_T2D])
R_residuals.index=R_residuals.index.astype(str)
R_residuals=R_residuals.loc[results['residuals'].index.astype(str)]

In [39]:
# Correlation matrices

R_corr_normal=pd.read_csv('R_corr_normal.csv', index_col=0)
R_corr_T2D=pd.read_csv('R_corr_T2D.csv', index_col=0)

In [40]:
# And the differential network

R_diff_net=pd.read_csv('R_diff_net.csv', index_col=0)

In [50]:
(results['residuals']-R_residuals).abs().max(axis=0).max()

0.00020221797813446685

In [51]:
(results['corr_ctrl']-R_corr_normal).abs().max().max()

3.6652569821854497e-06

In [52]:
(results['corr_t2d']-R_corr_T2D).abs().max().max()

2.5191664645811407e-07

In [53]:
(results['diff_net']-R_diff_net).abs().max().max()

3.661265512502898e-06